# Text-to-SVG V7: Compressed Data + Same-Session Inference

**NYU Deep Learning Spring 2026 — Kaggle Competition**

**Environment:** Google Colab Pro (A100 GPU) + Google Drive

### Setup Instructions
1. Upload `train.csv` and `test.csv` to your Google Drive under `MyDrive/svg-competition/`
2. Set Runtime → Change runtime type → **GPU** (A100 if available)
3. Run all cells in order
4. Adapter weights will be saved to `MyDrive/svg-competition/svg-lora-adapter/`

### What was wrong in V3-V5
1. Adapter keys didn't match between Unsloth (train) and PEFT (inference) — adapter silently ignored
2. Training SVGs had 15-digit floats, wasting model capacity on noise
3. No outlier removal — model tried to learn 7000+ char SVGs it could never reproduce

### V7 fixes
- **Same-session inference** — no adapter reloading, no key mismatch
- **SVG compression** — truncate floats, remove junk attrs (~60% smaller)
- **Outlier removal** — drop SVGs over 3000 chars / 30 paths after compression
- **No ET re-serialization** — keep training data in original format

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/v7_attention'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Contents: {os.listdir(PROJECT_DIR)}')

In [ ]:
!pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml cairosvg

In [ ]:
import os, re, time, random, json
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f'Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Configuration

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/v7_attention'

SYSTEM_PROMPT = 'Generate SVG code.'

ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter'
}

CONFIG = {
    'model_name': 'unsloth/Qwen3.5-2B',
    'max_seq_length': 2048,

    'lora_r': 32,
    'lora_alpha': 64,
    'lora_dropout': 0,

    'learning_rate': 1e-5,
    'num_train_epochs': 3,
    'per_device_train_batch_size': 8,
    'gradient_accumulation_steps': 4,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'max_grad_norm': 0.3,

    'logging_steps': 50,
    'save_steps': 300,
    'eval_steps': 300,
    'save_total_limit': 2,
    'output_dir': '/content/svg-lora-checkpoints',

    'train_csv': f'{PROJECT_DIR}/train.csv',
    'test_csv': f'{PROJECT_DIR}/test.csv',
    'eval_fraction': 0.02,
    'max_svg_chars': 16000,
    'adapter_save_dir': f'{PROJECT_DIR}/svg-lora-adapter-v7-attn',
}

for key in ['train_csv', 'test_csv']:
    print(f'{key}: {"FOUND" if os.path.exists(CONFIG[key]) else "NOT FOUND"}')

## 2. Data Cleaning: Compress SVGs + Remove Outliers

Key insight: training SVGs have 15-digit floats (e.g. `151.14999389648438`) that
can be truncated to 1 decimal (`151.1`) with zero visual difference. This cuts
SVG size by ~60%, making them learnable within the token budget.

Also remove: non-standard `filling=` attrs, default `fill-opacity=1.0`,
and outliers (too long/complex after compression).

In [ ]:
FLOAT_RE = re.compile(r'-?\d+\.\d{3,}')

def compress_svg(svg_text):
    """Compress SVG without changing visual appearance."""
    # Truncate long floats to 1 decimal place
    svg_text = FLOAT_RE.sub(
        lambda m: f'{float(m.group()):.1f}'.rstrip('0').rstrip('.'), svg_text
    )
    # Remove non-standard filling= attribute
    svg_text = re.sub(r'\s*filling="[^"]*"', '', svg_text)
    # Remove default fill-opacity=1.0
    svg_text = re.sub(r'\s*fill-opacity="1\.0"', '', svg_text)
    # Remove default stroke-opacity=1.0
    svg_text = re.sub(r'\s*stroke-opacity="1\.0"', '', svg_text)
    # Collapse whitespace
    svg_text = re.sub(r'\s+', ' ', svg_text).strip()
    return svg_text

def clean_training_svg(svg_text):
    """Compress + basic validation. No re-serialization through ET."""
    svg_text = svg_text.strip()
    if not svg_text.startswith('<svg'): return None
    if '</svg>' not in svg_text: return None

    # Compress
    svg_text = compress_svg(svg_text)

    # Outlier checks
    if len(svg_text) > 3000: return None          # Too complex after compression
    path_count = len(re.findall(r'<path[\s>]', svg_text))
    if path_count > 30: return None               # Too many paths

    return svg_text

# Load and clean
df = pd.read_csv(CONFIG['train_csv'])
print(f'Raw: {len(df)} rows')

valid_rows = []
reject_reasons = Counter()
original_lengths = []
compressed_lengths = []

for _, row in df.iterrows():
    svg = str(row['svg']).strip()
    prompt = str(row['prompt']).strip()
    if not prompt or len(prompt) < 5:
        reject_reasons['bad_prompt'] += 1; continue

    original_lengths.append(len(svg))
    cleaned = clean_training_svg(svg)
    if cleaned is None:
        reject_reasons['filtered'] += 1; continue

    compressed_lengths.append(len(cleaned))
    valid_rows.append({'prompt': prompt, 'svg': cleaned})

print(f'Clean: {len(valid_rows)} / {len(df)} ({100*len(valid_rows)/len(df):.1f}%)')
for r, c in reject_reasons.most_common(): print(f'  {r}: {c}')
print(f'\nOriginal SVG mean length: {sum(original_lengths)//len(original_lengths)}')
print(f'Compressed SVG mean length: {sum(compressed_lengths)//len(compressed_lengths)}')
print(f'Compression ratio: {100*(1 - sum(compressed_lengths)/sum(original_lengths)):.0f}%')

random.shuffle(valid_rows)
n_eval = max(100, int(len(valid_rows) * CONFIG['eval_fraction']))
train_dataset = Dataset.from_list(valid_rows[n_eval:])
eval_dataset = Dataset.from_list(valid_rows[:n_eval])
print(f'\nTrain: {len(train_dataset)} | Eval: {len(eval_dataset)}')

In [ ]:
def format_chat(example):
    return {'text': (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{example["prompt"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{example["svg"]}<|im_end|>'
    )}

train_formatted = train_dataset.map(format_chat, remove_columns=train_dataset.column_names)
eval_formatted = eval_dataset.map(format_chat, remove_columns=eval_dataset.column_names)

before = len(train_formatted)
train_formatted = train_formatted.filter(lambda x: len(x['text']) < CONFIG['max_seq_length'] * 3)
print(f'Train: {before} → {len(train_formatted)} after length filter')

## 3. Load Model + Train

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj'], # attention only
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps', eval_steps=CONFIG['eval_steps'],
    save_strategy='steps', save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    load_best_model_at_end=False,
    report_to='none', optim='paged_adamw_8bit', lr_scheduler_type='cosine',
    seed=SEED,
    max_length=CONFIG['max_seq_length'], packing=False, dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=train_formatted, eval_dataset=eval_formatted, args=sft_config,
)

eff_batch = CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']
print(f'Effective batch: {eff_batch} | Steps: {trainer.state.max_steps}')

In [ ]:
t0 = time.time()
train_result = trainer.train()
elapsed = (time.time() - t0) / 60
print(f'\nTraining complete in {elapsed:.1f} minutes')
print(f'Final train loss: {train_result.training_loss:.4f}')

## 4. Save Adapter (backup) + Switch to Inference

**Critical: do NOT reload the model.** Stay in the same session.

In [ ]:
# Save adapter as backup (in case Colab crashes before inference finishes)
adapter_dir = CONFIG['adapter_save_dir']
os.makedirs(adapter_dir, exist_ok=True)
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

summary = {
    'model': CONFIG['model_name'], 'lora_r': CONFIG['lora_r'],
    'lora_alpha': CONFIG['lora_alpha'], 'epochs': CONFIG['num_train_epochs'],
    'batch': eff_batch, 'lr': CONFIG['learning_rate'],
    'train_samples': len(train_formatted), 'final_loss': train_result.training_loss,
    'time_min': elapsed, 'seed': SEED,
}
with open(f'{PROJECT_DIR}/training_summary_v7.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Adapter backup saved: {adapter_dir}')
print(f'Summary: {summary}')

In [ ]:
# Switch to inference — SAME model, no reloading
FastLanguageModel.for_inference(model)
print('Switched to inference mode')

# Load text-only tokenizer to avoid vision model error during generate()
from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3.5-2B')
text_tokenizer.padding_side = 'left'
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token
    text_tokenizer.pad_token_id = text_tokenizer.eos_token_id
print('Text tokenizer loaded')

## 5. Verify Training Actually Changed the Model

In [ ]:
# The base Qwen3.5-2B wraps SVG in ```svg markdown blocks.
# If fine-tuning worked, output should be raw SVG without markdown.

messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': 'a red circle'}]
text = text_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = text_tokenizer(text, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False)

decoded = text_tokenizer.decode(out[0], skip_special_tokens=False)
assistant = decoded.split('<|im_start|>assistant\n')[-1]
if '</think>' in assistant: assistant = assistant.split('</think>')[-1]
for tok in ['<|im_end|>', '<|im_start|>', '<|endoftext|>']:
    assistant = assistant.split(tok)[0]

print('=== TRAINED MODEL OUTPUT ===')
print(assistant[:500])
print()
if '```' in assistant:
    print('⚠️ Still markdown-wrapped — training may not have taken effect')
elif '<svg' in assistant:
    print('✓ Raw SVG output — training is working!')
else:
    print('? Unexpected format')

## 6. Inference Functions

In [ ]:
SVG_REGEX = re.compile(r'<svg[\s\S]*?</svg>', flags=re.IGNORECASE)

def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ''

def fallback_svg(prompt):
    colors = ['red','blue','green','yellow','orange','purple','black','white','pink','brown','gray']
    fill = 'gray'
    for c in colors:
        if c in prompt.lower(): fill = c; break
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        f'<rect width="256" height="256" fill="white"/>'
        f'<circle cx="128" cy="128" r="64" fill="{fill}"/>'
        '</svg>'
    )

def strict_validate(svg):
    if not svg or len(svg) > 16000: return False
    try: root = ET.fromstring(svg)
    except ET.ParseError: return False
    root_tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
    if root_tag != 'svg': return False
    pc = 0
    for elem in root.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        if tag not in ALLOWED_TAGS: return False
        if tag == 'path': pc += 1
    return pc <= 256

def clean_and_validate_svg(svg):
    if not svg: return None
    ET.register_namespace('', 'http://www.w3.org/2000/svg')
    try: root = ET.fromstring(svg)
    except ET.ParseError: return None
    def remove_bad(elem):
        for child in list(elem):
            tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
            if tag not in ALLOWED_TAGS: elem.remove(child)
            else: remove_bad(child)
    remove_bad(root)
    root.set('width', '256'); root.set('height', '256')
    if 'viewBox' not in root.attrib: root.set('viewBox', '0 0 256 256')
    svg_out = ET.tostring(root, encoding='unicode')
    svg_out = svg_out.replace('ns0:', '').replace(':ns0', '')
    if 'xmlns=' not in svg_out:
        svg_out = svg_out.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    return svg_out if strict_validate(svg_out) else None

def generate_svg(prompt):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt}]
    text = text_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = text_tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
          **inputs, max_new_tokens=800,
          do_sample=False,              # Greedy — same as verification
          repetition_penalty=1.05,
    )

    decoded = text_tokenizer.decode(output_ids[0], skip_special_tokens=False)
    assistant = decoded.split('<|im_start|>assistant\n')[-1]
    if '</think>' in assistant: assistant = assistant.split('</think>')[-1]
    for tok in ['<|im_end|>', '<|im_start|>', '<|endoftext|>']:
        assistant = assistant.split(tok)[0]

    svg = extract_svg(assistant)
    if svg:
        result = clean_and_validate_svg(svg)
        if result: return result

    if '<svg' in assistant:
        start = assistant.index('<svg')
        partial = assistant[start:].rstrip()
        last_sc = partial.rfind('/>')
        last_et = partial.rfind('</')
        if last_et != -1:
            try: partial = partial[:partial.index('>', last_et) + 1]
            except ValueError:
                if last_sc != -1: partial = partial[:last_sc + 2]
        elif last_sc != -1:
            partial = partial[:last_sc + 2]
        if '</svg>' not in partial: partial += '</svg>'
        result = clean_and_validate_svg(partial)
        if result: return result

    return fallback_svg(prompt)

print('Inference functions ready.')

## 7. Quick Test

In [ ]:
for p in ['a red circle', 'blue star icon', 'green tree with brown trunk',
          'a simple house', 'five horizontal lines']:
    t1 = time.time()
    svg = generate_svg(p)
    valid = strict_validate(svg)
    fb = len(svg) < 190
    print(f'{time.time()-t1:.1f}s | len={len(svg)} | valid={valid} | fb={fb} | "{p}"')
    print(f'  {svg}')
    print()

## 8. Generate All 1000

In [ ]:
test_df = pd.read_csv(CONFIG['test_csv'])
print(f'Test: {len(test_df)} rows')

rows = []
fallback_count = 0
valid_count = 0
t0 = time.time()

for idx, row in test_df.iterrows():
    prompt = str(row['prompt']).strip()
    t1 = time.time()
    svg = generate_svg(prompt)
    gen_time = time.time() - t1
    is_fb = len(svg) < 190
    is_valid = strict_validate(svg)
    if is_fb: fallback_count += 1
    if is_valid: valid_count += 1
    rows.append({'id': row['id'], 'svg': svg})
    elapsed = time.time() - t0
    eta = elapsed / (idx + 1) * (len(test_df) - idx - 1) / 60
    print(f'  [{idx+1}/{len(test_df)}] {gen_time:.1f}s | len={len(svg)} | valid={is_valid} | fb={is_fb} | fb_total={fallback_count} | valid_total={valid_count} | ETA={eta:.0f}min')

elapsed_total = (time.time() - t0) / 60
print(f'\nDone! {len(rows)} SVGs in {elapsed_total:.1f} min')
print(f'Valid: {valid_count}/{len(rows)} ({100*valid_count/len(rows):.1f}%)')
print(f'Fallbacks: {fallback_count}/{len(rows)} ({100*fallback_count/len(rows):.1f}%)')

In [ ]:
sub_df = pd.DataFrame(rows)
SUBMISSION_PATH = f'{PROJECT_DIR}/submission_v7_attn.csv'
sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f'Saved: {SUBMISSION_PATH}')
print(f'SVG len mean={sub_df["svg"].str.len().mean():.0f} max={sub_df["svg"].str.len().max():.0f}')

from google.colab import files
files.download(SUBMISSION_PATH)

## AI Tooling Disclosure

- **Claude (Anthropic)**: Coding assistance, debugging.